## 1️⃣ IMPORTS & SETUP

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import joblib
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

from data_processing import DataProcessor

print('✅ All libraries imported successfully!')

## 2️⃣ CHARGER & RÉENTRAÎNER LE MODÈLE FINAL

In [ ]:
# Charger les données
data_path = '/Users/amayasbariz/Documents/dossier sans titre/projet ds/churn/data/customer_churn_business_dataset.csv'
processor = DataProcessor(data_path)
df = processor.load_data()
X, y = processor.get_X_y('churn')
numerical_features, categorical_features = processor.identify_features('churn')

print(f'✅ Données chargées: X{X.shape}, y{y.shape}')
print(f'   Churn rate: {y.sum()/len(y)*100:.2f}%')

In [ ]:
# 🔧 Prétraitement identique

# Gérer valeurs manquantes
for col in numerical_features:
    if X[col].isnull().sum() > 0:
        X[col].fillna(X[col].median(), inplace=True)

for col in categorical_features:
    if X[col].isnull().sum() > 0:
        X[col].fillna(X[col].mode()[0], inplace=True)

# Preprocessing
X_processed = X.copy()
scaler = StandardScaler()
X_processed[numerical_features] = scaler.fit_transform(X_processed[numerical_features])

label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    X_processed[col] = le.fit_transform(X_processed[col].astype(str))
    label_encoders[col] = le

print(f'✅ Prétraitement complété')
print(f'   Scaler: StandardScaler')
print(f'   Encoders: {len(label_encoders)} features catégorielles')

In [ ]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

# 🤖 Entraîner Random Forest (meilleur modèle)
print('🚀 Entraînement du Random Forest (modèle final)...')
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# Prédictions
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f'✅ Modèle entraîné!')
print(f'   Accuracy: {accuracy:.4f}')
print(f'   ROC-AUC: {roc_auc:.4f}')

## 3️⃣ SAUVEGARDER LE MODÈLE ET LES PRÉPROCESSEURS

In [ ]:
# 📁 Dossiers de sauvegarde
import os
os.makedirs('../models', exist_ok=True)

# 🔑 1. SAUVEGARDER LE MODÈLE
model_path = '../models/random_forest_model.joblib'
joblib.dump(rf_model, model_path)
print(f'✅ Modèle sauvegardé: {model_path}')

# 🔑 2. SAUVEGARDER LE SCALER
scaler_path = '../models/scaler.joblib'
joblib.dump(scaler, scaler_path)
print(f'✅ Scaler sauvegardé: {scaler_path}')

# 🔑 3. SAUVEGARDER LES LABEL ENCODERS
encoders_path = '../models/label_encoders.joblib'
joblib.dump(label_encoders, encoders_path)
print(f'✅ Label Encoders sauvegardés: {encoders_path}')

# 🔑 4. SAUVEGARDER LES FEATURES (IMPORTANT!)
feature_names_path = '../models/feature_names.json'
feature_info = {
    'all_features': list(X_processed.columns),
    'numerical_features': numerical_features,
    'categorical_features': categorical_features,
    'n_features': len(X_processed.columns)
}

with open(feature_names_path, 'w') as f:
    json.dump(feature_info, f, indent=2)

print(f'✅ Feature names sauvegardés: {feature_names_path}')

# 🔑 5. SAUVEGARDER LES MÉTADONNÉES DU MODÈLE
metadata = {
    'model_type': 'RandomForestClassifier',
    'n_estimators': 100,
    'training_date': datetime.now().isoformat(),
    'accuracy': float(accuracy),
    'roc_auc': float(roc_auc),
    'test_set_size': len(X_test),
    'n_features': len(X_processed.columns),
    'n_classes': 2,
    'class_names': ['No Churn', 'Churn']
}

metadata_path = '../models/model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'✅ Métadonnées sauvegardées: {metadata_path}')

print('\n' + '='*70)
print('✅ TOUS LES FICHIERS SAUVEGARDÉS AVEC SUCCÈS!')
print('='*70)

## 4️⃣ VÉRIFIER & CHARGER LE MODÈLE SAUVEGARDÉ

In [ ]:
# 🔍 Vérifier les fichiers sauvegardés
import os

print('🔍 FICHIERS SAUVEGARDÉS:\n')
for filename in os.listdir('../models'):
    filepath = f'../models/{filename}'
    size = os.path.getsize(filepath) / 1024  # Taille en KB
    print(f'   ✅ {filename:30s} - {size:8.2f} KB')

print('\n' + '='*70)
print('✅ TOUS LES FICHIERS PRÉSENTS!')

In [ ]:
# 🔄 CHARGER & TESTER LE MODÈLE SAUVEGARDÉ

print('🔄 CHARGEMENT DU MODÈLE DEPUIS LE DISQUE...\n')

# 1. Charger le modèle
loaded_model = joblib.load('../models/random_forest_model.joblib')
print('   ✅ Modèle chargé')

# 2. Charger le scaler
loaded_scaler = joblib.load('../models/scaler.joblib')
print('   ✅ Scaler chargé')

# 3. Charger les encoders
loaded_encoders = joblib.load('../models/label_encoders.joblib')
print(f'   ✅ Label Encoders chargés ({len(loaded_encoders)} features)')

# 4. Charger les features
with open('../models/feature_names.json', 'r') as f:
    loaded_feature_info = json.load(f)
print(f'   ✅ Feature names chargés ({loaded_feature_info["n_features"]} features)')

# 5. Charger les métadonnées
with open('../models/model_metadata.json', 'r') as f:
    loaded_metadata = json.load(f)
print(f'   ✅ Métadonnées chargées')

print('\n' + '='*70)
print('📊 MÉTADONNÉES DU MODÈLE')
print('='*70)
for key, value in loaded_metadata.items():
    if key not in ['training_date']:
        print(f'{key:20s}: {value}')
    else:
        print(f'{key:20s}: {value.split("T")[0]}')

In [ ]:
# 🧪 TEST: Faire une prédiction avec le modèle chargé

print('🧪 TEST DE PRÉDICTION AVEC LE MODÈLE CHARGÉ\n')

# Utiliser le test set
y_pred_loaded = loaded_model.predict(X_test)
y_pred_proba_loaded = loaded_model.predict_proba(X_test)[:, 1]

from sklearn.metrics import accuracy_score, roc_auc_score
accuracy_loaded = accuracy_score(y_test, y_pred_loaded)
roc_auc_loaded = roc_auc_score(y_test, y_pred_proba_loaded)

print(f'Résultats du modèle ORIGINAL:')
print(f'   Accuracy: {accuracy:.4f}')
print(f'   ROC-AUC:  {roc_auc:.4f}')

print(f'\nRésultats du modèle CHARGÉ:')
print(f'   Accuracy: {accuracy_loaded:.4f}')
print(f'   ROC-AUC:  {roc_auc_loaded:.4f}')

# Vérifier que c'est identique
diff_accuracy = abs(accuracy - accuracy_loaded)
diff_roc = abs(roc_auc - roc_auc_loaded)

if diff_accuracy < 1e-10 and diff_roc < 1e-10:
    print(f'\n✅ PARFAIT! Les modèles sont identiques (différence < 1e-10)')
else:
    print(f'\n⚠️ Différences détectées:')
    print(f'   Accuracy diff: {diff_accuracy}')
    print(f'   ROC-AUC diff:  {diff_roc}')

In [ ]:
# 🎯 EXEMPLE D'UTILISATION EN PRODUCTION

print('🎯 EXEMPLE D\'UTILISATION EN PRODUCTION\n')
print('Code à utiliser dans l\'API/Dashboard:\n')

example_code = '''
import joblib
import json
import pandas as pd

# Charger les artifacts
model = joblib.load('models/random_forest_model.joblib')
scaler = joblib.load('models/scaler.joblib')
encoders = joblib.load('models/label_encoders.joblib')

with open('models/feature_names.json') as f:
    feature_info = json.load(f)

# Préparer les données d'entrée (X_new)
X_new_processed = X_new.copy()
X_new_processed[feature_info['numerical_features']] = \
    scaler.transform(X_new_processed[feature_info['numerical_features']])

for col in feature_info['categorical_features']:
    X_new_processed[col] = encoders[col].transform(X_new_processed[col])

# Faire la prédiction
prediction = model.predict_proba(X_new_processed)[0, 1]  # Probabilité de churn
churn_prediction = model.predict(X_new_processed)[0]
'''

print(example_code)

print('\n' + '='*70)
print('✅ MODÈLE SAUVEGARDÉ ET PRÊT POUR LA PRODUCTION!')
print('='*70)